In [1]:
! pip install sqlalchemy

### MJODULO 8
#### LABORATORIO

In [2]:
from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

# Conexión SQLite
engine = create_engine("sqlite:///app.db")

# Base ORM
Base = declarative_base()


# Modelo
class User(Base):
    __tablename__ = "users"

    id = Column(Integer, primary_key=True)

    name = Column(String)


# CREAR TABLAS
Base.metadata.create_all(engine)

# Crear sesión
Session = sessionmaker(bind=engine)

session = Session()

# Insertar usuario
user = User(name="Fernando")

session.add(user)

session.commit()

print("Usuario insertado")

Usuario insertado


### MJODULO 9
#### LABORATORIO

In [3]:
# =========================================================
# LABORATORIO FASTAPI
# CRUD Orders + JWT + Testing
# =========================================================

# Instalar dependencias
# Ejecutar solo una vez
#! pip install fastapi uvicorn sqlalchemy python-jose passlib[bcrypt] httpx pytest


from typing import List

from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import OAuth2PasswordBearer
from jose import JWTError, jwt
from pydantic import BaseModel

# =========================================================
# CONFIG
# =========================================================

SECRET_KEY = "mysecretkey"
ALGORITHM = "HS256"

app = FastAPI()

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")

# =========================================================
# BASE DE DATOS FAKE EN MEMORIA
# =========================================================

orders_db = []

# =========================================================
# MODELOS PYDANTIC
# =========================================================


class OrderIn(BaseModel):
    product: str
    quantity: int


class OrderOut(BaseModel):
    id: int
    product: str
    quantity: int


class LoginRequest(BaseModel):
    username: str
    password: str


# =========================================================
# JWT
# =========================================================


def create_token(username: str):
    token = jwt.encode({"sub": username}, SECRET_KEY, algorithm=ALGORITHM)

    return token


def verify_token(token: str):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

        return payload

    except JWTError:
        raise HTTPException(status_code=401, detail="Token inválido")


async def get_current_user(token: str = Depends(oauth2_scheme)):
    return verify_token(token)


# =========================================================
# LOGIN
# =========================================================


@app.post("/login")
async def login(data: LoginRequest):
    # Validación fake
    if data.username != "admin" or data.password != "1234":
        raise HTTPException(status_code=401, detail="Credenciales inválidas")

    token = create_token(data.username)

    return {"access_token": token, "token_type": "bearer"}


# =========================================================
# CRUD ORDERS
# =========================================================


@app.post("/orders", response_model=OrderOut)
async def create_order(order: OrderIn, user=Depends(get_current_user)):
    new_order = {
        "id": len(orders_db) + 1,
        "product": order.product,
        "quantity": order.quantity,
    }

    orders_db.append(new_order)

    return new_order


@app.get("/orders", response_model=List[OrderOut])
async def get_orders(user=Depends(get_current_user)):
    return orders_db


@app.put("/orders/{order_id}", response_model=OrderOut)
async def update_order(order_id: int, order: OrderIn, user=Depends(get_current_user)):
    for o in orders_db:
        if o["id"] == order_id:
            o["product"] = order.product
            o["quantity"] = order.quantity

            return o

    raise HTTPException(status_code=404, detail="Order no encontrada")


@app.delete("/orders/{order_id}")
async def delete_order(order_id: int, user=Depends(get_current_user)):
    for o in orders_db:
        if o["id"] == order_id:
            orders_db.remove(o)

            return {"message": "Order eliminada"}

    raise HTTPException(status_code=404, detail="Order no encontrada")


# =========================================================
# ENDPOINT PROTEGIDO
# =========================================================


@app.get("/profile")
async def profile(user=Depends(get_current_user)):
    return {"user": user}


# =========================================================
# TESTING SIMPLE
# =========================================================

from fastapi.testclient import TestClient

client = TestClient(app)


def test_login():
    response = client.post("/login", json={"username": "admin", "password": "1234"})

    assert response.status_code == 200

    print(" Login test OK")


def test_create_order():
    # Login
    login_response = client.post(
        "/login", json={"username": "admin", "password": "1234"}
    )

    token = login_response.json()["access_token"]

    # Crear order
    response = client.post(
        "/orders",
        headers={"Authorization": f"Bearer {token}"},
        json={"product": "Laptop", "quantity": 2},
    )

    assert response.status_code == 200

    print("Create order test OK")


# Ejecutar tests
test_login()
test_create_order()

print("\n Laboratorio cargado correctamente")


# =========================================================
# CORRER API
# =========================================================

#
# uvicorn.run(app, host="0.0.0.0", port=8000)
#
# Swagger:
# http://127.0.0.1:8000/docs
# =========================================================

 Login test OK
Create order test OK

 Laboratorio cargado correctamente


MODULO 10
LABORATORIO

In [4]:
#! pip install hypothesis
#! pip install pytest-cov
import hypothesis.strategies as st
from hypothesis import given


def calculate_tax(total):
    return total * 1.16


# TEST
@given(st.floats(min_value=0))
def test_tax_positive(x):
    assert calculate_tax(x) >= x

MODULO 11
LABORATORIO

In [5]:
#! pip install requests
import time

import requests

urls = ["https://httpbin.org/delay/1"] * 5

start = time.time()

for url in urls:
    requests.get(url)

print(time.time() - start)

9.379414558410645


In [6]:
import asyncio
import time

import httpx

urls = ["https://httpbin.org/delay/1"] * 5


async def fetch(client, url):
    response = await client.get(url)

    return response.status_code


async def main():
    async with httpx.AsyncClient() as client:
        tasks = [fetch(client, u) for u in urls]

        results = await asyncio.gather(*tasks)

        print(results)


start = time.time()

await main()

print(time.time() - start)

[200, 200, 200, 200, 200]
2.2326173782348633


Modulo 12 
Laboratorio

In [7]:
# ==========================================
# LABORATORIO SOLID + DIP + PROTOCOLS
# Ports & Adapters (Hexagonal Architecture)
# ==========================================

from dataclasses import dataclass
from typing import Protocol

# ==========================================
# MODELO
# ==========================================


@dataclass
class User:
    name: str
    age: int


# ==========================================
# PUERTO / ABSTRACCIÓN
# ==========================================


class UserRepository(Protocol):
    def save(self, user: User) -> None: ...

    def get_all(self) -> list[User]: ...


# ==========================================
# IMPLEMENTACIÓN EN MEMORIA
# ==========================================


class InMemoryUserRepository:
    def __init__(self):
        self.users = []

    def save(self, user: User) -> None:
        self.users.append(user)

        print(f"[MEMORY] Usuario guardado: {user.name}")

    def get_all(self) -> list[User]:
        return self.users


# ==========================================
# IMPLEMENTACIÓN SQL (SIMULADA)
# ==========================================


class SQLUserRepository:
    def save(self, user: User) -> None:
        print(f"[SQL] INSERT INTO users (name, age) VALUES ('{user.name}', {user.age})")

    def get_all(self) -> list[User]:
        print("[SQL] SELECT * FROM users")

        return []


# ==========================================
# FACTORY
# ==========================================


class RepositoryFactory:
    @staticmethod
    def create(repo_type: str) -> UserRepository:
        if repo_type == "memory":
            return InMemoryUserRepository()

        elif repo_type == "sql":
            return SQLUserRepository()

        raise ValueError("Repositorio no soportado")


# ==========================================
# SERVICIO DESACOPLADO
# ==========================================


class UserService:
    def __init__(self, repository: UserRepository):
        self.repository = repository

    def register_user(self, name: str, age: int):
        if age < 0:
            raise ValueError("Edad inválida")

        user = User(name=name, age=age)

        self.repository.save(user)

    def list_users(self):
        return self.repository.get_all()


# ==========================================
# USO CON MEMORIA
# ==========================================

memory_repo = RepositoryFactory.create("memory")

memory_service = UserService(memory_repo)

memory_service.register_user("Fernando", 30)
memory_service.register_user("Ana", 25)

print("\nUsuarios en memoria:")

for user in memory_service.list_users():
    print(user)


# ==========================================
# USO CON SQL
# ==========================================

sql_repo = RepositoryFactory.create("sql")

sql_service = UserService(sql_repo)

sql_service.register_user("Carlos", 40)

print("\nConsulta SQL:")

sql_service.list_users()

[MEMORY] Usuario guardado: Fernando
[MEMORY] Usuario guardado: Ana

Usuarios en memoria:
User(name='Fernando', age=30)
User(name='Ana', age=25)
[SQL] INSERT INTO users (name, age) VALUES ('Carlos', 40)

Consulta SQL:
[SQL] SELECT * FROM users


[]

MODULO 13 
LABORATORIO

In [8]:
class PremiumStrategy:
    def calculate(self, price):
        return price * 0.7


cache = {}


def cached(func):
    def wrapper(x):
        if x not in cache:
            cache[x] = func(x)

        return cache[x]

    return wrapper


class StripeAPI:
    def make_payment(self):
        pass


class StripeAdapter:
    def __init__(self, stripe):
        self.stripe = stripe

    def pay(self):
        return self.stripe.make_payment()

MODULO 14
LABORATORIO

In [9]:
! pip install polars
! pip install scikit-learn pandas

In [10]:
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

data = load_iris()

# Features
X = data.data

y = data.target


# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Modelo
model = RandomForestClassifier()

model.fit(X_train, y_train)

# Predicción
preds = model.predict(X_test)

# Métrica
score = accuracy_score(y_test, preds)

print(score)

# Guardar
joblib.dump(model, "modelo.pkl")

0.9333333333333333


['modelo.pkl']

MODULO 16
LABORATORIO

In [ ]:
@dataclass
class Order:
    product: str

    quantity: int


@dataclass
class OrderCreated:
    product: str


class CreateOrder:
    def __init__(self, repository, uow, presenter):
        self.repository = repository

        self.uow = uow

        self.presenter = presenter

    def execute(self, product, quantity):
        order = Order(product=product, quantity=quantity)

        self.repository.save(order)

        OrderCreated(product=product)

        self.uow.commit()

        return self.presenter.present(order)

MODULO 17
LABORATORIO

In [12]:
! pip install build

In [13]:
! python -m build

* Creating isolated environment: venv+pip...
ERROR Source c:\Users\user\Documents\AXITY\laboratorios\NOTEBOOKS does not appear to be a Python project: no pyproject.toml or setup.py
